# pythscribe — the same `@wasm` kernel in Streamlit

Streamlit apps run via the `streamlit run` CLI (they can't be embedded inline in a notebook cell
the way Gradio's `demo.launch()` can), so this notebook **measures the `@wasm` speedup inline** and
then **launches a Streamlit app in a browser tab** where the *same* kernel runs **in the page** via
`pythscribe.streamlit.WasmComponent` (a sandboxed iframe; the compiled `.wasm` is base64'd into the
component args). The app shows the browser result plus live server-side speed metrics.

Run in the **PythScribe (Gradio)** kernel (`pyths-gradio`), which also has `streamlit` installed.

## The `@wasm` kernel + its speedup (measured here)

In [1]:
from __future__ import annotations
import random, time
from pythscribe import wasm, binding_of

@wasm
def pairwise(xs: list[float]) -> float:
    n = len(xs)
    total = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            d = xs[i] - xs[j]
            if d < 0.0:
                d = -d
            total = total + d
    return total

def pairwise_py(xs):
    n = len(xs); total = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            d = xs[i] - xs[j]
            if d < 0.0: d = -d
            total = total + d
    return total

rng = random.Random(0)
xs = [rng.random() for _ in range(400)]
pairwise(xs)   # warm (compile-on-first-call)

def med(fn, k=7):
    t = []
    for _ in range(k):
        t0 = time.perf_counter(); fn(); t.append((time.perf_counter() - t0) * 1e3)
    return sorted(t)[len(t) // 2]

tw = med(lambda: pairwise(xs)); tp = med(lambda: pairwise_py(xs))
print(f'pairwise (n=400)  ran as: {binding_of(pairwise).mode}')
print(f'plain Python : {tp:7.1f} ms')
print(f'server @wasm : {tw:7.1f} ms   speedup x{tp / tw:.0f}   (== plain Python bit-for-bit: {pairwise(xs) == pairwise_py(xs)})')

pairwise (n=400)  ran as: server
plain Python :     8.1 ms
server @wasm :     0.7 ms   speedup x12   (== plain Python bit-for-bit: True)


## Launch the Streamlit app (opens a browser tab)

This starts `streamlit run _streamlit_demo_app.py` as a background process. In the tab: move the
slider, click **Run @wasm in the browser** — the kernel runs *in the page* (`path=browser-wasm`,
`python_calls=0`), and the app shows the server-side `@wasm` vs plain-Python metrics. (Launching via
a subprocess also sidesteps the Windows/Jupyter event-loop limitation — `streamlit run` is a fresh
process.)

In [2]:
import socket, subprocess, sys, time
from pathlib import Path

APP = (Path('..') / '..').resolve() / 'examples/wasm-use-cases/_streamlit_demo_app.py'
PORT = 8501

_st_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', str(APP),
     '--server.port', str(PORT), '--server.headless', 'false',
     '--browser.gatherUsageStats', 'false'],
)
# wait until the server accepts connections
_up = False
for _ in range(40):
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            _up = True; break
    except OSError:
        time.sleep(0.5)
print(f'Streamlit {"running" if _up else "did not start in time"} at http://localhost:{PORT}'
      + ('  — a browser tab should open.' if _up else '  (check the process output).'))

Streamlit running at http://localhost:8501  — a browser tab should open.


## Stop the app when you're done

In [3]:
_st_proc.terminate()
try:
    _st_proc.wait(timeout=5)
except Exception:
    _st_proc.kill()
print('streamlit stopped')

streamlit stopped


## Recap

- The **same `@wasm` kernel** runs **in the browser** inside Streamlit (via `WasmComponent`), exactly
  as it does in Gradio — write it once, run it in either framework and on the server.
- The measured metric above (server `@wasm` vs plain Python) is the speedup that same kernel gives
  server-side; in the browser it computes with **no server round-trip** (`path=browser-wasm`).

See `gradio_demo.ipynb` (image downscale, 3 modes) and `gradio_more_demos.ipynb` (interactive
filters + zero-round-trip calculator) for the Gradio side; `full_features_demo.ipynb` for the full
speed / memory / capability tables.